# Ejemplos Nueva Arquitectura (Actualizado)

Este notebook muestra el flujo completo de la nueva arquitectura del Portfolio Tracker: inicialización, enriquecimiento de datos y simulación fiscal.

## 1. Inicializar Portfolio desde Excel

La clase `Portfolio` carga el Excel de Órdenes, lo parsea, y aplica el sistema FIFO automáticamente para guardar tus 'Lotes Abiertos'.

In [ ]:
import sys, os
import pandas as pd
from IPython.display import display

# Resolver rutas dinámicamente según desde dónde se lance el notebook
if os.path.exists('../app'):
    sys.path.append(os.path.abspath('..'))
    path = '../data/Ordenes.xlsx'
else:
    sys.path.append(os.path.abspath('./backend'))
    path = './backend/data/Ordenes.xlsx'

from app.services.core_portfolio import Portfolio

portfolio = Portfolio(path)

print("✅ Portfolio inicializado con éxito.")
print(f"Total de lotes abiertos gestionados: {len(portfolio.get_open_lots())}")

## 2. Mostrar Posiciones Activas

Extraemos la valoración general usando `portfolio.get_positions()`, que devuelve un DataFrame con el precio actual extraído en el momento.

In [ ]:
print("Obteniendo valoraciones de las posiciones...")
df_posiciones = portfolio.get_positions()
display(df_posiciones)

print(f"\nTotal de activos diferentes en cartera: {len(df_posiciones)}")
print(f"Capital Total Invertido (Coste de adquisición): {portfolio.get_total_invested():,.2f} €")

## 3. Enriquecer los datos financieros

Utilizamos la clase `Fund` en modo *detailed* para traer métricas avanzadas (asset allocation, regiones, precios en vivo) de Morningstar. Finalmente inyectamos esos precios a nuestro Portfolio para tener un DataFrame completo.

In [ ]:
from app.services.functions_fund import Fund

print("Enriqueciendo datos financieros de la cartera (Detailed Mode)...")
precios_actuales = {}
datos_detallados = {}

# Extraemos solo las 3 primeras posiciones para que no tarde mucho en el ejemplo
isins_ejemplo = list(portfolio.positions.keys())[:3]

for isin in isins_ejemplo:
    fund = Fund(isin=isin, mode="detailed", use_cache=True)
    df = fund.fund_data
    
    if df is not None and not df.empty:
        ms_data = df['data'].iloc[0]
        nombre = ms_data.get('name', isin)
        
        precio = ms_data.get('precio_actual', 0.0)
        if isinstance(precio, pd.Series): precio = precio.iloc[0]
        
        # Guardamos el precio para el portfolio global
        precios_actuales[isin] = float(precio) if pd.notna(precio) else 0.0
        
        # Extraemos métricas para visualización
        rv = ms_data.get('perc_Variable', [0]).iloc[0] if 'perc_Variable' in ms_data else 0
        rf = ms_data.get('perc_Fija', [0]).iloc[0] if 'perc_Fija' in ms_data else 0
        usa = ms_data.get('perc_region_northAmerica', [0]).iloc[0] if 'perc_region_northAmerica' in ms_data else 0
        
        datos_detallados[nombre] = {
            'Precio Actual': f"{precio:.2f} €",
            'Renta Variable': f"{rv:.1f}%",
            'Renta Fija': f"{rf:.1f}%",
            'EEUU (Norteamérica)': f"{usa:.1f}%"
        }

print("\n--- Métricas Enriquecidas (Morningstar) ---")
display(pd.DataFrame.from_dict(datos_detallados, orient='index'))

print("\n--- Cartera en formato DataFrame (usando to_dataframe) ---")
df_portfolio = portfolio.to_dataframe(live_prices=precios_actuales)
# Mostramos solo los fondos de ejemplo para no sobrecargar la vista
display(df_portfolio[df_portfolio['ISIN'].isin(isins_ejemplo)])

## 4. Calculadora Fiscal (Retirar 50.000€)

El `TaxOptimizer` analizará nuestros lotes abiertos, obtendrá los precios en vivo internamente, y nos sugerirá qué participaciones exactas debemos vender para aflorar la **menor** ganancia patrimonial posible, minimizando el impacto fiscal.

In [ ]:
from app.services.tax_calculator import TaxOptimizer

cantidad_a_retirar = 50000
print(f"Simulando una retirada optimizada de {cantidad_a_retirar:,}€...")

optimizer = TaxOptimizer(portfolio)
plan = optimizer.optimize_withdrawal(cantidad_a_retirar)

print("\n=== Plan de Retirada ===")
print(f"Cantidad Bruta Retirada: {plan['withdrawn_amount']:,.2f} €")
print(f"Ganancia Patrimonial Aflorada: {plan['total_capital_gain']:,.2f} €")
print(f"Impuestos Estimados a Pagar: {plan['estimated_tax']:,.2f} €")
print(f"Cantidad Neta en tu cuenta: {plan['net_amount']:,.2f} €\n")

print("Ventas recomendadas (Lotes específicos):")
for step in plan['plan']:
    fecha_str = step['Fecha_Compra'].strftime('%Y-%m-%d')
    print(f"- Vender {step['Participaciones_Vendidas']:.2f} parts. del fondo {step['Fondo']} (Compradas el {fecha_str}) -> Obtienes {step['Importe_Retirado']:.2f} €")